In [99]:
import pandas as pd

df = pd.read_csv("../data/processed/audible_clean.csv")

df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

print("Dataset 1 Shape:", df.shape)
print("Dataset 2 Shape:", df_advance.shape)

Dataset 1 Shape: (5439, 5)
Dataset 2 Shape: (4293, 9)


In [100]:
print("Dataset 1 Columns:")
print(df.columns.tolist())

print("\nDataset 2 Columns:")
print(df_advance.columns.tolist())

Dataset 1 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price']

Dataset 2 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Listening_Time_Minutes']


In [101]:
df_advance[[
    "Book Name",
    "Author",
    "Description",
    "Ranks and Genre"
]].head(5)

,Book Name,Author,Description,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,"Over the past three years, Jay Shetty has beco...",",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Brought to you by Penguin.,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,"In this generation-defining self-help guide, a...",",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,Brought to you by Penguin.,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,"Stop going through life, Start growing throug...",",#6 in Audible Audiobooks & Originals (See Top..."


In [102]:
# Display complete genre/rank information for 10 books
for value in df_advance["Ranks and Genre"].head(10):
    print(value)
    print("-" * 80)

,#1 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Personal Success,#1 in Stress Management,#2 in Society & Culture (Books)
--------------------------------------------------------------------------------
,#2 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Meditation (Audible Audiobooks & Originals),#1 in Self-Esteem,#2 in Personal Success
--------------------------------------------------------------------------------
,#3 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#3 in Personal Success,#16 in Personal Development & Self-Help
--------------------------------------------------------------------------------
,#5 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Psychology (Audible Audiobooks & Originals),#3 in Stress Management,#4 in Personal Success
--------------------------------------------------------------------------------
,#

In [103]:
import re

def extract_genres(rank_genre):
    if pd.isna(rank_genre):
        return []

    genres = re.findall(r"#\d+\s+in\s+([^,]+)", rank_genre)

    # Remove the overall Audible ranking
    genres = [
        genre.strip()
        for genre in genres
        if "Audible Audiobooks & Originals" not in genre
    ]

    return genres

df_advance["Genres"] = df_advance["Ranks and Genre"].apply(extract_genres)

df_advance[["Book Name", "Genres"]].head(10)

,Book Name,Genres
0,Think Like a Monk: The Secret of How to Harnes...,"[Personal Success, Stress Management, Society ..."
1,Ikigai: The Japanese Secret to a Long and Happ...,"[Self-Esteem, Personal Success]"
2,The Subtle Art of Not Giving a F*ck: A Counter...,"[Personal Success, Personal Development & Self..."
3,Atomic Habits: An Easy and Proven Way to Build...,"[Stress Management, Personal Success]"
4,Life's Amazing Secrets: How to Find Balance an...,"[Literary Essays, Essays (Books), Spiritualism]"
5,Sapiens,"[History of Civilization, Anthropology (Books)]"
6,Extraordinary Leadership,"[Leadership, Business Careers]"
7,The Intelligent Investor Rev Ed.,"[Personal Finance, Investing & Trading, Analys..."
8,Rich Dad Poor Dad: What the Rich Teach Their K...,"[Personal Finance, Investing & Trading, Person..."
9,The 5AM Club: Own Your Morning. Elevate Your L...,"[Sleep Disorders, Personal Success, Healthy Li..."


In [104]:
# Combine Description and Genres
df_advance["Content"] = (
    df_advance["Description"].fillna("") + " " +
    df_advance["Genres"].apply(lambda x: " ".join(x))
)

# Display the combined content
df_advance[["Book Name", "Content"]].head(5)

,Book Name,Content
0,Think Like a Monk: The Secret of How to Harnes...,"Over the past three years, Jay Shetty has beco..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Brought to you by Penguin. Self-Esteem Persona...
2,The Subtle Art of Not Giving a F*ck: A Counter...,"In this generation-defining self-help guide, a..."
3,Atomic Habits: An Easy and Proven Way to Build...,Brought to you by Penguin. Stress Management P...
4,Life's Amazing Secrets: How to Find Balance an...,"Stop going through life, Start growing throug..."


In [105]:
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words="english")

# Convert book content into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df_advance["Content"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (4293, 14732)


In [106]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Matrix Shape: ", cosine_sim.shape)

Cosine Similarity Matrix Shape:  (4293, 4293)


In [107]:
def recommend_book(book_name, cosine_sim=cosine_sim):

    # Find the book's positional index
    matches = np.flatnonzero(
        df_advance["Book Name"].eq(book_name).to_numpy()
    )

    if len(matches) == 0:
        print("Book not found in the dataset.")
        return

    idx = int(matches[0])

    # Get similarity scores
    similarity_scores = cosine_sim[idx]

    # Get top 6 indices (including the selected book)
    book_indices = np.argsort(similarity_scores)[::-1]

    # Exclude the selected book
    book_indices = [
        i for i in book_indices if i != idx
    ][:5]

    # Return recommendations
    recommendations = df_advance.iloc[book_indices][
        ["Book Name", "Author", "Rating", "Genres"]
    ].copy()

    recommendations["Similarity Score"] = [
        round(float(similarity_scores[i]) * 100, 2)
        for i in book_indices
    ]

    return recommendations.reset_index(drop=True)

In [108]:

# Test the recommendation system

book_name = df_advance["Book Name"].iloc[0]
print("Selected Book:", book_name)
recommendations = recommend_book(book_name)

print("\nTop 5 Recommended Books:")
display(recommendations)


Selected Book: Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now

Top 5 Recommended Books:


,Book Name,Author,Rating,Genres,Similarity Score
0,"The Classic Tales Podcast, Season Five",Agatha Christie,NaN,"[Caribbean & Latin American Literature, English]",15.88
1,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,"[Stress Management, Personal Success]",15.19
2,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,4.4,"[Social Media, Technology & Society, E-Commerce]",14.39
3,Ten Arguments For Deleting Your Social Media A...,Jaron Lanier,4.4,"[Social Media, Popular Culture Studies, Techno...",13.38
4,Social Media Marketing Workbook 2020,Jason McDonald PhD,4.5,"[Social Media, Internet & Social Media (Books)]",13.32


In [109]:

# Test invalid book title

recommendations = recommend_book("Unknown Book XYZ")


Book not found in the dataset.


Evaluation of Recommandation systen

In [110]:

# Evaluate Top 5 recommendations using Cosine Similarity

book_name = df_advance["Book Name"].iloc[0]

indices = pd.Series(
    df_advance.index,
    index=df_advance["Book Name"]
).drop_duplicates()

idx = indices[book_name]

similarity_scores = list(enumerate(cosine_sim[idx]))

similarity_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

similarity_scores = similarity_scores[1:6]

book_indices = [i[0] for i in similarity_scores]
scores = [i[1] for i in similarity_scores]

evaluation_results = df_advance.iloc[book_indices][
    ["Book Name", "Author", "Rating", "Genres"]
].copy()

evaluation_results["Similarity Score"] = scores

print("Selected Book:", book_name)

display(evaluation_results)


Selected Book: Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now


,Book Name,Author,Rating,Genres,Similarity Score
1865,"The Classic Tales Podcast, Season Five",Agatha Christie,NaN,"[Caribbean & Latin American Literature, English]",0.158788
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,"[Stress Management, Personal Success]",0.151893
1295,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,4.4,"[Social Media, Technology & Society, E-Commerce]",0.143922
448,Ten Arguments For Deleting Your Social Media A...,Jaron Lanier,4.4,"[Social Media, Popular Culture Studies, Techno...",0.133815
1043,Social Media Marketing Workbook 2020,Jason McDonald PhD,4.5,"[Social Media, Internet & Social Media (Books)]",0.133168


In [111]:

# Calculate Average Similarity Score

average_similarity = evaluation_results["Similarity Score"].mean()

print("Average Similarity Score:", round(average_similarity, 4))
print("Average Similarity Percentage:", round(average_similarity * 100, 2), "%")


Average Similarity Score: 0.1443
Average Similarity Percentage: 14.43 %


In [112]:

# Select a book with a known title

book_name = "Atomic Habits: An Easy and Proven Way to Build Good Habits and Break Bad Ones"

recommendations = recommend_book(book_name)

display(recommendations)


,Book Name,Author,Rating,Genres,Similarity Score
0,The Stress Solution: The 4 Steps to Reset Your...,Dr Rangan Chatterjee,4.7,[Stress Management],87.86
1,The Antidote: Happiness for People Who Can't S...,Oliver Burkeman,4.5,[Personal Success],74.49
2,Surrounded by Idiots: The Four Types of Human ...,Thomas Erikson,4.5,[Personal Success],74.49
3,Happy: Why More or Less Everything Is Absolute...,Derren Brown,4.4,[Personal Success],74.49
4,"Die Trying: Jack Reacher, Book 2",Lee Child,4.4,[],57.17


In [113]:

# Calculate Precision@5

relevant_books = 4
k = 5

precision_at_5 = relevant_books / k

print("Precision@5:", round(precision_at_5, 2))
print("Precision@5 Percentage:", round(precision_at_5 * 100, 2), "%")


Precision@5: 0.8
Precision@5 Percentage: 80.0 %


In [114]:
# Test recommendations for multiple audiobooks

test_books = df_advance["Book Name"].dropna().drop_duplicates().sample(
    5, random_state=42
)

for book in test_books:
    print("\nSelected Book:", book)
    print("-" * 80)

    recommendations = recommend_book(book)

    display(recommendations)


Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,The Soccer Mystery: The Boxcar Children Myster...,Gertrude Chandler Warner,5.0,[],17.65
1,"Coaching for Performance, Fourth Edition: GROW...",John Whitmore,4.5,"[Mentoring & Coaching, Leadership]",15.86
2,Learn Hindi - Level 1: Introduction to Hindi: ...,Innovative Language Learning LLC,NaN,[],13.76
3,The Third Door: The Wild Quest to Uncover How ...,Alex Banayan,4.7,"[Mentoring & Coaching, Personal Success in Bus...",11.19
4,"The Science Fiction Hall of Fame, Vol. 2-B",Ben Bova,4.7,[],11.12



Selected Book: Leadership on the Line (Revised): Staying Alive Through the Dangers of Change
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,"[Personal Success, Stress Management, Society ...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,[],0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,[],0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,[],0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,[],0.0



Selected Book: The Disaster Artist
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,"[Personal Success, Stress Management, Society ...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,[],0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,[],0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,[],0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,[],0.0



Selected Book: The Gravity Between Us
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,"[Personal Success, Stress Management, Society ...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,[],0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,[],0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,[],0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,[],0.0



Selected Book: 101 Animals Stories
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,"[Personal Success, Stress Management, Society ...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,[],0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,[],0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,[],0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,[],0.0


In [115]:

print("Cosine Similarity Shape:", cosine_sim.shape)
print("Data Type:", type(cosine_sim))
print("Selected Row Shape:", cosine_sim[0].shape)


Cosine Similarity Shape: (4293, 4293)
Data Type: <class 'numpy.ndarray'>
Selected Row Shape: (4293,)


In [116]:

book_name = test_books.iloc[0]

indices = pd.Series(
    range(len(df_advance)),
    index=df_advance["Book Name"]
).drop_duplicates()

idx = indices[book_name]

print("Selected Book:", book_name)
print("Book Index:", idx)
print("Index Type:", type(idx))
print("Similarity Row Shape:", cosine_sim[idx].shape)


Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
Book Index: 1307
Index Type: <class 'numpy.int64'>
Similarity Row Shape: (4293,)


In [117]:
import numpy as np
book_name = test_books.iloc[0]

matches = np.flatnonzero(
    df_advance["Book Name"].eq(book_name).to_numpy()
)

print("Selected Book:", book_name)
print("Matching Positions:", matches)
print("Number of Matches:", len(matches))

if len(matches) > 0:
    idx = int(matches[0])
    print("Selected Index:", idx)
    print("Index Type:", type(idx))
    print("Similarity Row Shape:", cosine_sim[idx].shape)

Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
Matching Positions: [1307]
Number of Matches: 1
Selected Index: 1307
Index Type: <class 'int'>
Similarity Row Shape: (4293,)


In [118]:
similarity_row = cosine_sim[idx]

print("Data Type:", similarity_row.dtype)
print("Shape:", similarity_row.shape)

print("\nNaN Values:", np.isnan(similarity_row).sum())
print("Infinite Values:", np.isinf(similarity_row).sum())

print("\nMinimum:", np.min(similarity_row))
print("Maximum:", np.max(similarity_row))

print("\nFirst 10 Values:")
print(similarity_row[:10])

print("\nUnique Values Count:", len(np.unique(similarity_row)))

Data Type: float64
Shape: (4293,)

NaN Values: 0
Infinite Values: 0

Minimum: 0.0
Maximum: 1.0000000000000002

First 10 Values:
[0.02037953 0.         0.03259895 0.         0.01778682 0.0143836
 0.         0.         0.04287459 0.01544431]

Unique Values Count: 987


In [119]:
similarity_scores = list(enumerate(cosine_sim[idx]))

print("Similarity Scores Type:", type(similarity_scores))
print("First Item:", similarity_scores[0])
print("First Item Type:", type(similarity_scores[0]))

print("\nSecond Element Type:", type(similarity_scores[0][1]))
print("Second Element Shape:", np.shape(similarity_scores[0][1]))

print("\nChecking All Scores:")

for i, score in similarity_scores:
    if np.ndim(score) != 0:
        print("Non-scalar score found!")
        print("Index:", i)
        print("Score:", score)
        print("Shape:", np.shape(score))
        break
else:
    print("All similarity scores are scalar values.")

Similarity Scores Type: <class 'list'>
First Item: (0, np.float64(0.020379528111855144))
First Item Type: <class 'tuple'>

Second Element Type: <class 'numpy.float64'>
Second Element Shape: ()

Checking All Scores:
All similarity scores are scalar values.


In [120]:
print("Total Books:", len(df_advance))

print("\nEmpty Content:")
print(df_advance["Content"].isna().sum())

print("\nBlank Content:")
print(
    df_advance["Content"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("\nContent Length Statistics:")
print(df_advance["Content"].fillna("").str.len().describe())

print("\nBooks with Shortest Content:")
display(
    df_advance.assign(
        Content_Length=df_advance["Content"].fillna("").str.len()
    ).nsmallest(10, "Content_Length")[
        ["Book Name", "Genres", "Content", "Content_Length"]
    ]
)

Total Books: 4293

Empty Content:
0

Blank Content:
1056

Content Length Statistics:
count    4293.000000
mean      223.697181
std       227.875991
min         1.000000
25%        21.000000
50%       208.000000
75%       316.000000
max      2226.000000
Name: Content, dtype: float64

Books with Shortest Content:


,Book Name,Genres,Content,Content_Length
48,The Hard Thing About Hard Things: Building a B...,[],,1
51,"Men Are from Mars, Women Are from Venus: The C...",[],,1
57,Hit Refresh,[],,1
58,A Suitable Boy (Dramatised),[],,1
59,Take Control of Your Life: How to Silence Fear...,[],,1
60,The 10X Rule: The Only Difference Between Succ...,[],,1
61,A Game of Thrones: Book 1 of A Song of Ice and...,[],,1
144,Breath: The New Science of a Lost Art,[],,1
145,Origin,[],,1
146,Chinta Chhodo Sukh Se Jiyo,[],,1


In [121]:
# Replace missing descriptions and genres with empty strings
df_advance["Description"] = df_advance["Description"].fillna("").astype(str)
df_advance["Genres"] = df_advance["Genres"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else str(x)
)

# Build improved recommendation content
df_advance["Content"] = (
    df_advance["Description"] + " " +
    df_advance["Genres"] + " " +
    df_advance["Book Name"].fillna("") + " " +
    df_advance["Author"].fillna("")
).str.strip()

# Check updated content
print("Total Books:", len(df_advance))

print(
    "Blank Content:",
    df_advance["Content"].str.strip().eq("").sum()
)

print("\nUpdated Content Examples:")
display(
    df_advance[
        ["Book Name", "Author", "Genres", "Content"]
    ].head(10)
)

Total Books: 4293
Blank Content: 0

Updated Content Examples:


,Book Name,Author,Genres,Content
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,Personal Success Stress Management Society & C...,"Over the past three years, Jay Shetty has beco..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Self-Esteem Personal Success,Brought to you by Penguin. Self-Esteem Persona...
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,Personal Success Personal Development & Self-Help,"In this generation-defining self-help guide, a..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,Stress Management Personal Success,Brought to you by Penguin. Stress Management P...
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,Literary Essays Essays (Books) Spiritualism,"Stop going through life, Start growing throug..."
5,Sapiens,Yuval Noah Harari,History of Civilization Anthropology (Books),Fire gave us power. Farming made us hungry for...
6,Extraordinary Leadership,Robin Sharma,Leadership Business Careers,Leadership Business Careers Extraordinary Lead...
7,The Intelligent Investor Rev Ed.,Benjamin Graham,Personal Finance Investing & Trading Analysis ...,The Classic Text Annotated to Update Graham's ...
8,Rich Dad Poor Dad: What the Rich Teach Their K...,Robert T. Kiyosaki,Personal Finance Investing & Trading Personal ...,Rich Dad Poor Dad will…. Personal Finance Inve...
9,The 5AM Club: Own Your Morning. Elevate Your L...,Robin Sharma,Sleep Disorders Personal Success Healthy Livin...,Legendary leadership and elite performance exp...


In [122]:
import pandas as pd

df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

print("Dataset Loaded Successfully!")
print("Shape:", df_advance.shape)
print("Columns:", df_advance.columns.tolist())

Dataset Loaded Successfully!
Shape: (4293, 9)
Columns: ['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Listening_Time_Minutes']


In [135]:
df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

In [136]:
# Rebuild Content Column

df_advance["Genres"] = df_advance["Ranks and Genre"].fillna("").astype(str)

df_advance["Content"] = (
    df_advance["Description"].fillna("").astype(str) + " " +
    df_advance["Genres"] + " " +
    df_advance["Book Name"].fillna("").astype(str) + " " +
    df_advance["Author"].fillna("").astype(str)
).str.strip()

# Save updated dataset
df_advance.to_csv(
    "../data/processed/audible_advanced_clean.csv",
    index=False
)

print("Updated dataset saved successfully!")
print("Dataset Shape:", df_advance.shape)
print("Content Column Exists:", "Content" in df_advance.columns)
print("Blank Content:", df_advance["Content"].eq("").sum())

Updated dataset saved successfully!
Dataset Shape: (4293, 11)
Content Column Exists: True
Blank Content: 0


In [137]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = tfidf.fit_transform(df_advance["Content"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

TF-IDF Matrix Shape: (4293, 10000)
Cosine Similarity Matrix Shape: (4293, 4293)


In [138]:
print("Dataset Shape:", df_advance.shape)

print("\nAvailable Columns:")
print(df_advance.columns.tolist())

print("\nContent Column Exists:",
      "Content" in df_advance.columns)

Dataset Shape: (4293, 11)

Available Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Listening_Time_Minutes', 'Genres', 'Content']

Content Column Exists: True


In [ ]:
import numpy as np

In [133]:
# Test recommendations for multiple audiobooks

test_books = df_advance["Book Name"].dropna().drop_duplicates().sample(
    5, random_state=42
)

for book in test_books:
    print("\nSelected Book:", book)
    print("-" * 80)

    recommendations = recommend_book(book)

    display(recommendations)


Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,The Soccer Mystery: The Boxcar Children Myster...,Gertrude Chandler Warner,5.0,,17.65
1,"Coaching for Performance, Fourth Edition: GROW...",John Whitmore,4.5,",#1,764 in Audible Audiobooks & Originals (See...",15.86
2,Learn Hindi - Level 1: Introduction to Hindi: ...,Innovative Language Learning LLC,NaN,", 5 star, , (0%), ...",13.76
3,The Third Door: The Wild Quest to Uncover How ...,Alex Banayan,4.7,",#2,711 in Audible Audiobooks & Originals (See...",11.19
4,"The Science Fiction Hall of Fame, Vol. 2-B",Ben Bova,4.7,,11.12



Selected Book: Leadership on the Line (Revised): Staying Alive Through the Dangers of Change
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,",#1 in Audible Audiobooks & Originals (See Top...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,,0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,,0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,,0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,,0.0



Selected Book: The Disaster Artist
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,",#1 in Audible Audiobooks & Originals (See Top...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,,0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,,0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,,0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,,0.0



Selected Book: The Gravity Between Us
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,",#1 in Audible Audiobooks & Originals (See Top...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,,0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,,0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,,0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,,0.0



Selected Book: 101 Animals Stories
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,",#1 in Audible Audiobooks & Originals (See Top...",0.0
1,The Disease Delusion: Conquering the Causes of...,Dr. Jeffrey S. Bland,4.7,,0.0
2,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,,0.0
3,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,,0.0
4,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,,0.0
